# 中文文本摘要微調：以 mT5 實作 2026 標準 Seq2Seq 流程

## 學習目標

本 notebook 以中文新聞摘要為任務，示範如何以 Seq2Seq 架構微調預訓練模型。完成後你將能夠：

1. 理解 Seq2Seq 任務的資料處理流程（encoder 端輸入 → decoder 端目標）
2. 使用現代 `TrainingArguments`（bf16、warmup、cosine scheduler、AdamW fused）
3. 以 `evaluate.load("rouge")` + `compute_metrics` 正確評估中文摘要品質
4. 掌握以標準 HF Seq2Seq API 進行可攜、可維護的模型前處理與推論流程

## 前置知識

- 已完成 `02-Adv-tasks/01-text_classification/` 系列（了解 Trainer 基本用法）
- 了解 Seq2Seq 架構基本概念（encoder-decoder）

## 與相鄰 notebook 的銜接

- 上一個：`../06-question_answering/` — Extractive QA（span prediction）
- 下一個：`../08-machine_translation/` — 翻譯任務，同樣是 Seq2Seq 架構

## 為什麼選用 mT5？

本 notebook 使用 `google/mt5-small`（多語言 T5），支援 101 種語言（含繁/簡體中文），其 tokenizer 完全遵循標準 HF Seq2Seq 介面，與 TRL、PEFT 等工具鏈無縫整合，換模型時無需重寫前處理程式碼。

## 版本鎖定與環境確認

統一安裝版本確保可重現性。若在已配置好的環境中執行，可跳過此 cell。

In [ ]:
# Version pinning — 2026 unified requirements
# Run once; skip if environment is already configured.
!pip install -q \
    "transformers>=4.46" \
    "datasets>=3.0" \
    "accelerate>=1.0" \
    "evaluate>=0.4" \
    "sentencepiece>=0.1.99" \
    "safetensors>=0.4" \
    "torch>=2.4" \
    "rouge-score>=0.1.2"

In [ ]:
import transformers, datasets, evaluate, torch

print(f"transformers : {transformers.__version__}")
print(f"datasets     : {datasets.__version__}")
print(f"evaluate     : {evaluate.__version__}")
print(f"torch        : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 1 — 匯入套件

In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
from datasets import load_from_disk, load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)
import evaluate

# Fix seed for reproducibility across all random sources
set_seed(42)

## Step 2 — 載入資料集

### 資料說明：NLPCC 2017 中文摘要

NLPCC 2017 單文件摘要任務使用中文新聞資料，每筆樣本包含：
- `content`：新聞正文
- `title`：對應的人工摘要標題

### 路徑策略

1. 優先透過環境變數 `NLPCC_DATA_DIR` 指定
2. 若無環境變數，回退到 notebook 同目錄下的 `nlpcc_2017/`
3. 未來可替換為 `load_dataset("your-org/nlpcc2017")` 一行搞定

In [ ]:
# Resolve dataset path via environment variable or local fallback
_data_env = os.environ.get("NLPCC_DATA_DIR", "")
DATA_DIR = Path(_data_env) if _data_env else Path("./nlpcc_2017")

print(f"Loading dataset from: {DATA_DIR.resolve()}")
ds_full = load_from_disk(str(DATA_DIR))
print(ds_full)

### 訓練 / 測試分割

使用明確的關鍵字引數 `test_size=100` 指定測試集大小，確保語意清晰、跨版本行為一致。

In [ ]:
# Explicit keyword argument prevents ambiguity across dataset versions
if isinstance(ds_full, DatasetDict):
    # Already split (e.g., loaded from HF Hub)
    ds = ds_full
else:
    ds = ds_full.train_test_split(test_size=100, seed=42)

print(ds)
print("\nSample record:")
print(ds["train"][0])

## Step 3 — 載入 Tokenizer 與模型

### 為什麼選用 mT5？

`google/mt5-small` 是多語言 T5（mT5），支援 101 種語言（含繁/簡體中文），使用完全標準的 HF Seq2Seq API，可直接與 TRL / PEFT 等工具鏈整合，換模型時無需重寫前處理程式碼。

### 裝置與精度策略（2026 統一慣例）

```python
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    device_map='auto',        # 自動分配 GPU/CPU/disk
    torch_dtype=torch.bfloat16,  # bf16 優於 fp16：不容易 overflow，訓練更穩定
    use_safetensors=True,     # 比 pickle 安全、載入速度快約 30%
)
```

**bf16 vs fp16 vs fp32**：
- `fp32`：精度最高，但顯存佔用是 bf16 的 2 倍
- `fp16`：顯存減半，但指數範圍小，容易在梯度累積時溢位（overflow → NaN）
- `bf16`：指數範圍與 fp32 相同（只犧牲尾數精度），對訓練穩定性友善；需要 Ampere（A100/3090）或更新架構

**VRAM 參考（mT5 系列，bf16）**：
- `mt5-small`（300M）：~0.6 GB
- `mt5-base`（580M）：~1.2 GB
- `mt5-large`（1.2B）：~2.4 GB

In [ ]:
# Model selection — mt5-small fits in <2 GB VRAM; upgrade to mt5-base or mt5-large as needed
MODEL_ID = "google/mt5-small"  # ~300M params, ~0.6 GB VRAM in bf16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

print(f"Tokenizer class : {type(tokenizer).__name__}")
print(f"Vocab size      : {tokenizer.vocab_size:,}")
print(f"Model max length: {tokenizer.model_max_length}")

In [ ]:
# 2026 unified loading convention
# device_map='auto'  → Accelerate decides GPU/CPU/disk offload automatically
# torch_dtype=bfloat16 → stable training on Ampere+; set to float32 on older GPUs
# use_safetensors=True → avoids pickle, ~30% faster load, safer in shared environments

_bf16_supported = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
_dtype = torch.bfloat16 if _bf16_supported else torch.float32
print(f"Using dtype: {_dtype}  (bf16 supported: {_bf16_supported})")

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=_dtype,
    use_safetensors=True,
)

print(model.config.model_type)
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## Step 4 — 資料前處理

### Seq2Seq 標籤遮罩（-100）原理

Seq2Seq 訓練時，`labels` 中值為 `-100` 的位置不計入 loss（PyTorch CrossEntropyLoss 的 `ignore_index` 預設值）。因此 padding token 必須設為 `-100`，否則模型會試圖去預測 pad，浪費計算並污染梯度。

`DataCollatorForSeq2Seq` 在 batch 組裝時自動完成這個遮罩，不需手動處理。

### 為什麼用 `batched=True`？

`dataset.map(fn, batched=True)` 讓 tokenizer 一次處理多筆樣本（預設 batch=1000），利用向量化運算，速度通常快 3-5 倍。

### 前綴（prefix）設計

mT5 在微調時需要在輸入前加任務前綴，告訴模型要做什麼：
```
摘要生成: {正文}
```
這是 T5 系列的標準做法（參考原始 T5 論文），讓同一個模型可以區分不同任務。

In [ ]:
# Hyperparameters for tokenization
MAX_INPUT_LENGTH = 512    # encoder side — truncate long articles
MAX_TARGET_LENGTH = 64   # decoder side — max summary length
TASK_PREFIX = "摘要生成: "   # mT5 task prefix (T5-style multitask conditioning)


def preprocess_function(examples):
    """Tokenize source (content) and target (title) for Seq2Seq training.

    The tokenizer handles:
    - Prefix prepend for task conditioning
    - Truncation to MAX_INPUT_LENGTH on encoder side
    - Label encoding with truncation to MAX_TARGET_LENGTH
    """
    inputs = [TASK_PREFIX + text for text in examples["content"]]

    # Tokenize encoder inputs
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        # No padding here — DataCollatorForSeq2Seq will pad dynamically per batch
    )

    # Tokenize decoder targets using `text_target` parameter (standard Seq2Seq API)
    labels = tokenizer(
        text_target=examples["title"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


# batched=True processes 1000 samples at a time — ~3-5x faster than row-by-row
tokenized_ds = ds.map(
    preprocess_function,
    batched=True,
    remove_columns=ds["train"].column_names,
    desc="Tokenizing dataset",
)

print(tokenized_ds)
print(f"\nTrain sample keys: {list(tokenized_ds['train'][0].keys())}")

In [ ]:
# Verify a single example: decode input and label to confirm correctness
sample = tokenized_ds["train"][0]

print("=== Encoder input (decoded) ===")
print(tokenizer.decode(sample["input_ids"], skip_special_tokens=True)[:200])

print("\n=== Decoder target / label (decoded) ===")
# Filter out -100 placeholders before decoding
valid_labels = [t for t in sample["labels"] if t != -100]
print(tokenizer.decode(valid_labels, skip_special_tokens=True))

## Step 5 — 評測函數

### ROUGE 指標說明

| 指標 | 意義 |
|------|------|
| ROUGE-1 | 單詞（unigram）重疊率 |
| ROUGE-2 | 雙詞組（bigram）重疊率 |
| ROUGE-L | 最長公共子序列（句子流暢度） |

中文 ROUGE 計算需要在字元層級切分（或以空格分隔），而非以詞為單位，因為中文沒有天然的空格分詞。

`evaluate.load("rouge")` 是 HuggingFace 官方維護的實作，透過 `rouge_score` 後端支援多語言，並可直接接入 `Trainer` 的 `compute_metrics` 鉤子。

In [ ]:
rouge_metric = evaluate.load("rouge")


def compute_metrics(eval_preds):
    """Decode predictions and labels, then compute ROUGE scores.

    Called automatically by Seq2SeqTrainer at every evaluation step.
    """
    preds, labels = eval_preds

    # Replace -100 (ignored index) with pad_token_id before decoding
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # Decode token ids to strings
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Chinese ROUGE: split into individual characters with spaces
    # This ensures ROUGE treats each character as a token (no word segmentation needed)
    decoded_preds = [" ".join(list(p.strip())) for p in decoded_preds]
    decoded_labels = [" ".join(list(l.strip())) for l in decoded_labels]

    result = rouge_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=False,  # No stemming for Chinese
    )

    # Round for readability
    return {k: round(v, 4) for k, v in result.items()}

## Step 6 — 配置訓練參數

### 2026 TrainingArguments 關鍵設定解說

```python
bf16=True               # 使用 bfloat16 混合精度（需要 Ampere 或更新架構）
warmup_ratio=0.1        # 前 10% steps 線性升溫，防止初期梯度爆炸
lr_scheduler_type='cosine'  # Cosine decay 比 linear 更平滑，訓練後期不容易震盪
optim='adamw_torch_fused'   # PyTorch 2.x 的 fused AdamW，速度快 10-20%
max_grad_norm=1.0       # Gradient clipping，防止梯度爆炸
save_safetensors=True   # 儲存用 safetensors 格式取代 pickle
eval_strategy='steps'   # 每 N steps 評測一次，比 epoch 更細粒度
load_best_model_at_end=True  # 訓練結束後自動載回最佳 checkpoint
seed=42                 # 固定隨機種子確保可重現
```

**為什麼用 AdamW 而非 Adam？**

Adam 的 weight decay 實作有缺陷：它把 weight decay 混入自適應學習率的分母，等效於對大梯度參數施加較少的 decay。AdamW 把 weight decay 獨立出來（decoupled weight decay），對所有參數施加一致的 L2 正則化，訓練更穩定。

**Effective Batch Size**：
```
effective_batch = per_device_train_batch_size × gradient_accumulation_steps × num_gpus
                = 4 × 8 × 1 = 32
```

In [ ]:
OUTPUT_DIR = Path("./output/summary_mt5")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    # --- Batch & accumulation ---
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,   # effective batch = 4 x 8 = 32
    # --- Learning rate schedule ---
    learning_rate=5e-4,
    num_train_epochs=3,
    warmup_ratio=0.1,                # linear warmup for first 10% of steps
    lr_scheduler_type="cosine",      # smooth cosine decay
    # --- Precision & optimizer ---
    bf16=_bf16_supported,            # bfloat16 on Ampere+; falls back to fp32
    fp16=False,                      # explicitly disable fp16 to avoid conflict
    optim="adamw_torch_fused",       # PyTorch 2.x fused AdamW (~10-20% faster)
    max_grad_norm=1.0,               # gradient clipping
    # --- Logging & evaluation ---
    logging_steps=16,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    # --- Checkpoint & reproducibility ---
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    save_total_limit=2,              # keep only 2 checkpoints to save disk space
    save_safetensors=True,           # save in safetensors format
    seed=42,
    # --- Seq2Seq specific ---
    predict_with_generate=True,      # required for compute_metrics with generation
    generation_max_length=MAX_TARGET_LENGTH,
    report_to="none",                # disable W&B / MLflow in demo
)

print("TrainingArguments configured.")
print(f"Effective batch size: {args.per_device_train_batch_size * args.gradient_accumulation_steps}")

## Step 7 — 建立 DataCollator 與 Trainer

### DataCollatorForSeq2Seq 的作用

`DataCollatorForSeq2Seq` 在每個 mini-batch 組裝時動態 padding（而非在 `map` 階段固定 padding 到最大長度），好處是：

1. **節省計算**：同一 batch 的序列 pad 到該 batch 的最長序列，而非整體最長序列
2. **自動 -100 遮罩**：自動將 `labels` 中的 pad token 替換為 `-100`，使其不被計入 loss

In [ ]:
# DataCollatorForSeq2Seq:
# - Pads input_ids and attention_mask to the longest sequence in each batch
# - Pads labels and replaces pad tokens with -100 (ignore_index for CrossEntropyLoss)
collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,              # dynamic padding per batch
    label_pad_token_id=-100,   # mask padding in labels
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,  # ROUGE evaluation at each eval step
)

print("Trainer ready.")
print(f"Train samples : {len(tokenized_ds['train'])}")
print(f"Eval samples  : {len(tokenized_ds['test'])}")

## Step 8 — 模型訓練

> **VRAM 需求（mT5-small, bf16, batch=4, no quantization）**：約 3-4 GB GPU VRAM。  
> 若 VRAM 不足，可調低 `per_device_train_batch_size=2` 或改用 `google/mt5-small` 的量化版本。

In [ ]:
train_result = trainer.train()

# Log training metrics
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)

In [ ]:
# Save the best model and tokenizer in safetensors format
trainer.save_model(str(OUTPUT_DIR / "best_model"))
tokenizer.save_pretrained(str(OUTPUT_DIR / "best_model"))
print(f"Model saved to: {(OUTPUT_DIR / 'best_model').resolve()}")

## Step 9 — 評測（整個測試集）

使用 `trainer.evaluate()` 在整個測試集上計算 ROUGE。`compute_metrics` 已包含所有解碼與指標計算邏輯，無需手刻迴圈。

In [ ]:
eval_metrics = trainer.evaluate(
    eval_dataset=tokenized_ds["test"],
    metric_key_prefix="test",
)

trainer.log_metrics("test", eval_metrics)

print("\n=== ROUGE Scores on Test Set ===")
for k, v in eval_metrics.items():
    if "rouge" in k:
        print(f"  {k:20s}: {v:.4f}")

## Step 10 — 單筆推論示範

### 2026 標準推論寫法（可攜到任何 Seq2Seq 模型）

```python
inputs = tokenizer(TASK_PREFIX + text, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=MAX_TARGET_LENGTH, num_beams=4)
summary = tokenizer.decode(output[0], skip_special_tokens=True)
```

標準 HF Seq2Seq 介面只需三行：tokenize、generate、decode，不依賴任何模型專屬方法。

In [ ]:
def summarize(text: str, num_beams: int = 4) -> str:
    """Generate a summary for a single Chinese article.

    Uses beam search (num_beams=4) for higher quality output compared to greedy.
    """
    model.eval()
    inputs = tokenizer(
        TASK_PREFIX + text,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_TARGET_LENGTH,
            num_beams=num_beams,
            early_stopping=True,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


# Demo on the last test sample
sample = ds["test"][-1]
print("=== 原始文章（前 300 字）===")
print(sample["content"][:300])
print("\n=== 人工摘要 ===")
print(sample["title"])
print("\n=== 模型生成摘要 ===")
print(summarize(sample["content"]))

## Step 11 — 批次推論與 ROUGE 計算

以下示範直接在 Python 中對一批樣本計算 ROUGE，不依賴 Trainer，適合推論部署時的離線評估場景。

In [ ]:
# Batch inference on a subset of test samples for quick evaluation
EVAL_SUBSET = 40  # mirror the original notebook's subset size
test_subset = ds["test"].select(range(min(EVAL_SUBSET, len(ds["test"]))))

predictions = []
references = []

model.eval()
for i, sample in enumerate(test_subset):
    pred = summarize(sample["content"])
    ref = sample["title"]
    predictions.append(pred)
    references.append(ref)
    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(test_subset)} samples")

print(f"\nTotal predictions: {len(predictions)}")

In [ ]:
# Compute ROUGE with character-level tokenization for Chinese
decoded_preds_char = [" ".join(list(p.strip())) for p in predictions]
decoded_refs_char = [" ".join(list(r.strip())) for r in references]

rouge_result = rouge_metric.compute(
    predictions=decoded_preds_char,
    references=decoded_refs_char,
    use_stemmer=False,
)

print("=== Batch ROUGE Scores ===")
for k, v in rouge_result.items():
    print(f"  {k}: {v:.4f}")

print("\n=== Sample Predictions ===")
for i in range(min(3, len(predictions))):
    print(f"[{i}] Reference : {references[i]}")
    print(f"[{i}] Prediction: {predictions[i]}")
    print()

## Step 12 — 推送至 Hugging Face Hub（選用）

將訓練好的模型推送到 Hub，方便後續部署與分享。需要先執行 `huggingface-cli login`。

In [ ]:
# Optional: push to Hub
# Requires: huggingface-cli login (or HF_TOKEN environment variable)
#
# REPO_ID = "your-username/mt5-small-nlpcc2017-summarization"
#
# trainer.push_to_hub(
#     repo_id=REPO_ID,
#     commit_message="Train mt5-small on NLPCC 2017 Chinese summarization",
#     tags=["summarization", "chinese", "mt5", "seq2seq"],
#     language=["zh"],
#     license="apache-2.0",
# )
# print(f"Model pushed to: https://huggingface.co/{REPO_ID}")

print("Push to Hub is commented out. Uncomment and set REPO_ID to enable.")

## 小結

### 本 Notebook 完成的事

1. 以 `google/mt5-small` 實作標準 HF Seq2Seq 微調流程，tokenizer 介面完全可攜，換模型無需重寫前處理程式碼
2. 採用 2026 統一載入慣例：`device_map='auto'`、`torch_dtype=bfloat16`、`use_safetensors=True`
3. 以 `DataCollatorForSeq2Seq` 的動態 padding 節省計算資源，`labels` 中的 pad 自動替換為 `-100` 不計入 loss
4. 以 `evaluate.load("rouge")` 計算中文 ROUGE，整合進 `compute_metrics` 鉤子
5. `Seq2SeqTrainingArguments` 配置 `bf16`、`warmup_ratio`、`cosine` scheduler、`adamw_torch_fused`、`save_safetensors`

### 關鍵概念回顧

| 概念 | 要點 |
|------|------|
| Seq2Seq 標籤遮罩 | `labels` 中 `-100` 的位置不計入 loss，DataCollator 自動處理 |
| 任務前綴 | `"摘要生成: "` 讓 mT5 識別任務類型，是 T5 系列微調的標準做法 |
| 中文 ROUGE | 需在字元層級切分，用空格分隔後再計算 |
| bf16 vs fp16 | bf16 指數範圍與 fp32 相同，訓練更穩定；需要 Ampere+ GPU |
| 動態 padding | 每個 batch 只 pad 到該 batch 最長序列，比固定 pad 到 512 節省 |

### 延伸練習

1. 將模型換成 `google/mt5-base`（580M），觀察 ROUGE 分數提升幅度與訓練時間的取捨
2. 加入 `EarlyStoppingCallback`，在驗證 ROUGE-L 不再提升時提前終止
3. 試試在 `Seq2SeqTrainingArguments` 中調整 `generation_num_beams=4`，觀察 eval 時 beam search 對 ROUGE 的影響
4. 使用 PEFT + LoRA 只微調部分參數，達到同等 ROUGE 但訓練記憶體需求大幅降低（參考 `04-Fine-tuning/` 系列）